In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json, zipfile
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix
from baseline import calculate_resilience_cost
import matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)

# --- Load data ---
df = pd.read_csv("data/train.csv", index_col=0)
test_df = pd.read_csv("data/test.csv",  index_col=0)
cost_matrix_df = pd.read_csv("data/cost_matrix.csv", index_col=0)

display(df)

In [ ]:
display(cost_matrix_df.head())


In [ ]:

# Makes no sense to do it this way, ruins the ordinal structure, let's set it to green->yellow->orange->red
cost_matrix_df = cost_matrix_df.sort_values(by=["0"])
cost_matrix_df = cost_matrix_df[['0', '3', '1', '2']]
display(cost_matrix_df.head())
cost_matrix = cost_matrix_df.values

In [ ]:
# Let's rename just in case

In [ ]:
# Data inspection
display(df.describe())
df.info()


In [ ]:
# Let's change the types for efficiency (maybe it happens already automatically though..)


# is_magnitude_int = all(x.is_integer() for x in df["magnitude"])
# print("Is magnitude actually ints: ", is_magnitude_int)
# # Magnitude is proper float

# is_depth_int = all(x.is_integer() for x in df["depth"])
# print("Is depth actually ints: ", is_depth_int)
# # depth should be uint
# df.depth = df.depth.astype("UInt16")

# is_cdi_int = all(x.is_integer() for x in df["cdi"])
# print("Is cdi actually ints: ", is_cdi_int)
# # cdi should be uint
# df.cdi = df.cdi.astype("UInt8")

# is_mmi_int = all(x.is_integer() for x in df["mmi"])
# print("Is mmi actually ints: ", is_mmi_int)
# # mmi should be int
# df.mmi = df.mmi.astype("UInt8")

# is_sig_int = all(x.is_integer() for x in df["sig"])
# print("Is sig actually ints: ", is_sig_int)
# # sig should be int
# df.sig = df.sig.astype("Int16")

# This actually didn't do anything (also should be applied to test df anyway), it probably gets converted back to floats for computation anyway...

In [ ]:
df.info()

In [ ]:
df.alert = pd.Categorical(df.alert, ["green", "yellow", "orange", "red"], ordered=True)
df.sort_values(by=["alert"], inplace=True)
display(df.alert)
df.reset_index(drop=True, inplace=True)
len(df) - len(df.drop_duplicates())
df.info()
# No dups

In [ ]:

sns.barplot(data=df["alert"], estimator="size") 

In [ ]:

# sns.histplot(data=df, x = "magnitude", hue="alert",)
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="magnitude")


In [ ]:
sns.scatterplot(data=df, x=df.magnitude, y=df.index, hue=df.alert)

In [ ]:
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="depth")

In [ ]:
sns.scatterplot(data=df, x=df.depth, y=df.index, hue=df.alert)
# Some outliers for orange


In [ ]:
outliers = (df.alert == "orange") & (df.depth > 100)
df.drop(df[outliers].index, inplace=True)
print("After drops")
sns.scatterplot(data=df, x=df.depth, y=df.index, hue=df.alert)


In [ ]:
# sns.histplot(data=df["cdi"], hue=df["alert"])
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="cdi")


In [ ]:
plt.figure(figsize = (8, 10))

sns.scatterplot(data=df, x=df.cdi, y=df.index, hue=df.alert)


In [ ]:
# sns.histplot(data=df["mmi"])

sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="mmi")


In [ ]:
plt.figure(figsize = (10, 8))

sns.scatterplot(data=df, x=df.mmi, y=df.index, hue=df.alert)
# Looks like some outliers for everything except red

In [ ]:
# outliers = ((df.alert == "orange") & (df.mmi == 9)) | ((df.alert == "yellow") & (df.mmi == 9)) | ((df.alert == "green") & (df.mmi < 3)) # This one seems tricky on the performance
# df.drop(df[outliers].index, inplace=True)
sns.scatterplot(data=df, x=df.mmi, y=df.index, hue=df.alert)


In [ ]:

# sns.histplot(data=df["sig"])
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="sig")

In [ ]:
plt.figure(figsize = (10, 8))
sns.scatterplot(data=df, x=df.sig, y=df.index, hue=df.alert)


In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder ,OrdinalEncoder
from sklearn.metrics import make_scorer

categories=[["green", "yellow", "orange", "red"]]
color_to_int = {"green": 0, "yellow": 1, "orange": 2, "red": 3}
# --- Encode target and split ---
# enc = OrdinalEncoder(categories=categories, dtype=int, )
# enc = LabelEncoder()
print("Enc: ")
# display(enc)
# enc.fit(df.loc[:, ["alert"]])
print("Pre mapping y:")
display(df.alert)
y = df.alert.map(color_to_int).astype(int)
print("Post mapping y:")
display(y)
X = df.drop("alert", axis=1)
X_final = test_df
display(X.corr())
display(df.info())

int_to_color = {0: "green", 1: "yellow", 2: "orange", 3: "red"}
def mean_resilience_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    rci = calculate_resilience_cost(cm, cost_matrix)
    return rci / len(y_true)

# We use resilience_score rather than accuracy, since this is what will be competed upon
resilience_scorer = make_scorer(mean_resilience_score, greater_is_better=False, )

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import PowerTransformer

pipe = make_pipeline(StandardScaler())
numeric_features = ["magnitude", "depth", "cdi", "mmi", "sig"]
numeric_transformer = Pipeline(
    steps=[("scaler", StandardScaler())]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        # ("cat", categorical_transformer, categorical_features),
    ]
)

# numeric_log = ['depth'] # Strongly skewed to the right
# numeric_scale_only = ['magnitude', 'cdi', 'mmi', 'sig']

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('power', PowerTransformer(method='yeo-johnson'), numeric_log),
#         ('scale', StandardScaler(), numeric_scale_only),
#     ]
# )

X_train, X_val_test, y_train, y_val_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_val_test, y_val_test, test_size=0.5, random_state=SEED, stratify=y_val_test
)


## SVM

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.compose import TransformedTargetRegressor

class_weight = {
        0: 0.075,
        1: 0.35,
        2: 0.5,
        3: 0.125,
    }
class_weight = {
        0: 0.025,
        1: 0.075,
        2: 0.30,
        3: 0.6,
    }
clf = Pipeline(
    # steps=[("preprocessor", preprocessor), ("classifier", SVC(C=700, class_weight="balanced"))])
    steps=[("preprocessor", preprocessor), ("classifier", SVC(C=700, class_weight=class_weight))]
    # steps=[("preprocessor", preprocessor), ("classifier", LinearRegression())]
)
clf

In [ ]:
from matplotlib import pyplot as plt
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold





# The class weights are very interesting, they seem to make a big impact since recall can be slightly controlled 



outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

param_grid = {
    'classifier__gamma': ["scale"],
    'classifier__kernel': [ "rbf"],
    'classifier__C': np.arange(10, 100, 2)
    }
    # 'classifier_class_weigths_': [{"}
    # 'classifier__C': np.linspace(1, 1)}

grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    scoring=resilience_scorer,
    cv=inner_cv)

nested_scores = cross_val_score(grid_search, X, y, cv=outer_cv, scoring=resilience_scorer, )


In [ ]:

print(f"Outer CV Scores (Generalization Error): {nested_scores}")
print(f"Mean Nested Stratified CV Resilience Estimate: {np.mean(nested_scores):.4f}")

# Plotting the nested CV scores
plt.figure(figsize=(8, 5))
plt.bar(range(1, len(nested_scores) + 1), nested_scores, color='purple')
plt.axhline(np.mean(nested_scores), color='red', linestyle='--', label=f'Mean Nested Resilience Score ({np.mean(nested_scores):.4f})')
plt.title('Nested Stratified CV Scores (Tuning Error)')
plt.xlabel('Outer Fold Number')
plt.ylabel('Resilience Score')
plt.ylim(-0, -50)
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay




In [ ]:
grid_search.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
display(grid_search.best_params_)
display(grid_search.best_score_)
clf = grid_search

# --- Validate ---
y_test_pred = clf.predict(X_test)
f1 = f1_score(y_test, y_test_pred, average="macro")
cm = confusion_matrix(y_test, y_test_pred)
rci = calculate_resilience_cost(cm, cost_matrix)

print(classification_report(y_test, y_test_pred))

cm_display = ConfusionMatrixDisplay(cm, display_labels=["green", "yellow", "orange", "red"]).plot()
print(f"F1 (macro): {f1:.3f}")
# print("Confusion matrix:\n", cm)
print(f"Mean Resilience Cost: {rci / len(y_test):.2f}")

# GradientBoosting

In [ ]:

# from sklearn.ensemble import GradientBoostingClassifier
# from sklearn.preprocessing import PowerTransformer

# numeric_log = ['depth'] # Strongly skewed to the right
# numeric_scale_only = ['magnitude', 'cdi', 'mmi', 'sig']

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('power', PowerTransformer(method='yeo-johnson'), numeric_log),
#         ('scale', StandardScaler(), numeric_scale_only),
#     ]
# )

# clf_boosting = Pipeline(
#     steps=[("preprocessor", preprocessor), ("classifier", GradientBoostingClassifier())]
# )

# param_grid = {
#     "classifier__n_estimators": [150, 300],
#     "classifier__learning_rate": [0.03, 0.1],
#     "classifier__max_depth": [2, 3]
#     }

# mod_gbc = GridSearchCV(
#     estimator=clf_boosting,
#     param_grid=param_grid,
#     scoring=resilience_scorer,
#     cv=inner_cv)

# nested_scores = cross_val_score(mod_gbc, X, y, cv=outer_cv, scoring="accuracy" )


In [ ]:
# mod_gbc.fit(X_train, y_train)

## Neural Network

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, BatchNormalization
from keras.layers import Activation, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping
from keras import regularizers, Input
from keras.losses import sparse_categorical_crossentropy
from keras.utils import set_random_seed

import tensorflow as tf
set_random_seed(SEED)
cost_matrix = tf.constant(cost_matrix, dtype=tf.float32)

def cost_sensitive_loss(y_true, y_pred):
    y_true = tf.cast(tf.squeeze(y_true), tf.int32)          
    row = tf.gather(cost_matrix, y_true)                    
    loss = tf.reduce_sum(row * y_pred, axis=1)              
    return tf.reduce_mean(loss)

def hybrid_loss(alpha=0.5):
    def loss(y_true, y_pred):
        ce = sparse_categorical_crossentropy(y_true, y_pred)
        cs = cost_sensitive_loss(y_true, y_pred)
        return alpha * ce + (1 - alpha) * cs
    return loss

def create_ann():
    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(1024, activation='relu'),
        Dense(1024, activation='relu'),
        Dense(64, activation='relu'),
        Dense(4, activation='softmax')
    ])

    model.compile(optimizer='adam', loss=hybrid_loss(0.8), metrics=['accuracy'])
    return model
early_stopping_monitor = EarlyStopping(patience=30, min_delta=0.000, restore_best_weights=True, verbose=True)
early_stopping_monitor = EarlyStopping(patience=100, restore_best_weights=True, verbose=True)

mod_ann = create_ann()
mod_ann.summary()

history = mod_ann.fit(
    # pd.concat([X_train, X_val]),
    # pd.concat([y_train , y_val]),  
    # validation_data=(X_test,y_test),
    pd.concat([X_train]),
    pd.concat([y_train]),            
    validation_data=(X_val,y_val),
    shuffle = True,
    batch_size = int(16384 * 1024 * 2),
    epochs=2000,
    verbose=1,
    callbacks=[early_stopping_monitor],
    
)

In [ ]:
# print(mod_ann.get.get_metrics_result())



In [ ]:
# extract the loss and metric values
loss = history.history['loss']
acc = history.history['accuracy']
val_loss = history.history['val_loss']
val_acc = history.history['val_accuracy']

# plot the loss
plt.plot(loss, label='loss')
plt.plot(val_loss, label='val_loss')
plt.legend()
plt.show()

# plot the accuracy
plt.plot(acc, label='accuracy')
plt.plot(val_acc, label='val_accuracy')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report


y_test_pred_prob = mod_ann.predict(X_test)
y_test_pred = y_test_pred_prob.argmax(axis=1)

f1 = f1_score(y_test, y_test_pred, average="macro")
cm = confusion_matrix(y_test, y_test_pred)
rci = calculate_resilience_cost(cm, cost_matrix)

print(classification_report(y_test, y_test_pred))

cm_display = ConfusionMatrixDisplay(cm, display_labels=["green", "yellow", "orange", "red"]).plot()
print(f"F1 (macro): {f1:.3f}")
# print("Confusion matrix:\n", cm)
print(f"Mean Resilience Cost: {rci / len(y_test):.2f}")

In [ ]:
print(f"Outer CV Scores (Generalization Error): {nested_scores}")
print(f"Mean Nested Stratified CV Resilience Estimate: {np.mean(nested_scores):.4f}")

# Plotting the nested CV scores
plt.figure(figsize=(8, 5))
plt.bar(range(1, len(nested_scores) + 1), nested_scores, color='purple')
plt.axhline(np.mean(nested_scores), color='red', linestyle='--', label=f'Mean Nested Resilience Score ({np.mean(nested_scores):.4f})')
plt.title('Nested Stratified CV Scores (Tuning Error)')
plt.xlabel('Outer Fold Number')
plt.ylabel('Resilience Score')
plt.ylim(-0, 1)
plt.legend()
plt.show()



In [ ]:

# # # --- Train baseline model ---
# clf = KNeighborsClassifier(n_neighbors=10)
# clf.fit(X_train, y_train)
# clf


In [ ]:
from baseline import create_submission

# best_params = grid_search.best_params_
# final_clf = Pipeline(
#     steps=[
#         ("preprocessor", preprocessor),
#         ("classifier", SVC(
#             C=best_params["classifier__C"],
#             gamma=best_params["classifier__gamma"],
#             kernel=best_params["classifier__kernel"],
#             class_weight=class_weight
#         ))
#     ]
# )

# final_clf.fit(X, y)
y_final = mod_ann.predict(X_final).argmax(axis=1)
# display(best_params)
display([int_to_color[i] for i in y_final][1:3])
create_submission([int_to_color[i] for i in y_final])